# Fine tune SBERT 

In [1]:
import pandas as pd 

import pickle
import json
import os

from collections import Counter
from itertools import combinations 

from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sentence_transformers import InputExample, SentenceTransformer, losses, SentencesDataset, LoggingHandler, evaluation
from torch.utils.data import DataLoader

import logging

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks
from comments_saver import CommentsSaver

/opt/miniconda3/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open('../outputs/SBERT_pairs/train_examples.pkl', 'rb') as f:
    train_examples = pickle.load(f)

In [3]:
# select just 10% of train_examples
train_examples = train_examples[:int(len(train_examples) * 0.1)]

In [8]:
# summary stats of train_examples
print(f"Number of training examples: {len(train_examples)}")

Number of training examples: 890988


In [9]:
# Configure logging
logging.basicConfig(format='%(asctime)s - %(message)s',
                    level=logging.INFO)

In [10]:
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=32,
    num_workers=4  # Adjust based on your CPU
)

In [11]:
model = SentenceTransformer('all-MiniLM-L6-v2')

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model=model)

2025-05-28 09:17:45,006 - Use pytorch device_name: mps
2025-05-28 09:17:45,007 - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


In [13]:
model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=2)

Step,Training Loss
500,0.106900
1000,0.043100
1500,0.034400
2000,0.030200
2500,0.026900
3000,0.023900
3500,0.021700
4000,0.019100


KeyboardInterrupt: 

In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    output_path='output/my_model',
    show_progress_bar=True
)

ValueError: You have set `args.eval_strategy` to steps, but you didn't provide an `eval_dataset` or an `evaluator`. Either provide an `eval_dataset` or an `evaluator` to `SentenceTransformerTrainer`, or set `args.eval_strategy='no'` to skip evaluation.